#Ingest constructors.json file
1. Read the file using Spark DataFrame reader API
2. Add Metadata Columns
    - Source File
    - Ingestion TimeStamp
3. Write to bronze delta table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_path = f"{landing_folder_path}/{v_batch_id}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
source_path

### Step 1- Read the json file using the DataFrame Reader API

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DoubleType
constructors_schema = StructType([
            StructField('constructorId',StringType()),
            StructField('name',StringType()),
            StructField('nationality',StringType()),
            StructField('url',StringType())
])

# constructors_schema = """
#                  constructorId STRING,
#                  name STRING,
#                  nationality STRING
#                  url STRING
#                                   """

In [0]:
constructors_df = (
    spark.read
         .format("json")
         .option("header","true")
#        .option("inferSchema","true")
         .schema(constructors_schema)
         .option("mode","FAILFAST")
         .load(source_path)
        )


In [0]:
display(constructors_df)

###Step 2- Add Metadata Columns
- Source File
- Ingestion TimeStamp



In [0]:

constructors_final_df = add_ingestion_metadata(constructors_df)

In [0]:
display(constructors_final_df)

### Step 3- Write to bronze delta table

In [0]:
v_batch_id

In [0]:
write_to_bronze(
    input_df = constructors_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))